##Machine Learning

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [2]:
model_data = pd.read_parquet(
    "../data/processed/model_data.parquet"
)

In [3]:
X = model_data.drop(columns="icu_admission")
y = model_data["icu_admission"]

In [18]:
categorical_columns = [
    "sex",
    "race",
    "department",
    "antype",
    "icd10_pcs",
]

numeric_columns = ["age", "bmi", "asa"]

X[numeric_columns] = X[numeric_columns].replace(
    [np.inf, -np.inf],
    np.nan,
)

X[numeric_columns] = X[numeric_columns].fillna(
    X[numeric_columns].median()
)

In [19]:
X.shape

(130960, 2311)

In [20]:
X.columns[:20]

Index(['age', 'asa', 'emop', 'bmi', 'sex_M', 'department_CTS', 'department_DM',
       'department_EM', 'department_GS', 'department_IM', 'department_NS',
       'department_OG', 'department_OL', 'department_OS', 'department_OT',
       'department_PED', 'department_PS', 'department_RAD', 'department_RO',
       'department_UR'],
      dtype='object')

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [22]:
scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

##Model 1 - Logistic Regression

In [23]:
model = LogisticRegression(
    max_iter=3000,
    random_state=42,
)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,3000
,multi_class,'deprecated'


In [24]:
model.n_iter_

array([125], dtype=int32)

In [25]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [26]:
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.3f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob):.3f}")

Accuracy : 0.904
Precision: 0.786
Recall   : 0.472
F1 Score : 0.590
ROC AUC  : 0.887


##Model 2 - XGBoost

In [27]:
from xgboost import XGBClassifier

In [28]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
)

In [29]:
xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [30]:
xgb_pred = xgb_model.predict(X_test)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

In [31]:
print(f"Accuracy : {accuracy_score(y_test, xgb_pred):.3f}")
print(f"Precision: {precision_score(y_test, xgb_pred):.3f}")
print(f"Recall   : {recall_score(y_test, xgb_pred):.3f}")
print(f"F1 Score : {f1_score(y_test, xgb_pred):.3f}")
print(f"ROC AUC  : {roc_auc_score(y_test, xgb_prob):.3f}")

Accuracy : 0.906
Precision: 0.805
Recall   : 0.474
F1 Score : 0.596
ROC AUC  : 0.889


#Risk Category

In [32]:
def get_risk_category(probability):
    if probability < 0.30:
        return "Low"
    elif probability < 0.70:
        return "Medium"
    else:
        return "High"

In [33]:
example_probability = 0.82

get_risk_category(example_probability)

'High'

#Length of Hospital Stay and Mortality Rate

In [34]:
operations = pd.read_csv(
    "../data/raw/inspire/operations.csv.gz",
    compression="gzip",
)

In [35]:
operations["hospital_los_days"] = (
    operations["discharge_time"] - operations["admission_time"]
) / 1440

operations["inhospital_mortality"] = (
    operations["inhosp_death_time"]
    .notna()
    .astype(int)
)

In [36]:
extended_model_data = model_data.copy()

extended_model_data["hospital_los_days"] = operations[
    "hospital_los_days"
].values

extended_model_data["inhospital_mortality"] = operations[
    "inhospital_mortality"
].values

In [37]:
feature_columns = [
    column
    for column in extended_model_data.columns
    if column not in [
        "icu_admission",
        "hospital_los_days",
        "inhospital_mortality",
    ]
]

X_extended = extended_model_data[feature_columns]

In [38]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_los = extended_model_data["hospital_los_days"]

X_los_train, X_los_test, y_los_train, y_los_test = train_test_split(
    X_extended,
    y_los,
    test_size=0.2,
    random_state=42,
)

In [41]:
feature_columns = [
    column
    for column in extended_model_data.columns
    if column not in [
        "icu_admission",
        "hospital_los_days",
        "inhospital_mortality",
    ]
]

X_extended = extended_model_data[feature_columns].copy()

In [42]:
categorical_columns = [
    "sex",
    "race",
    "department",
    "antype",
    "icd10_pcs",
]

X_extended = pd.get_dummies(
    X_extended,
    columns=categorical_columns,
    drop_first=True,
    dtype=int,
)

In [43]:
X_extended = X_extended.replace(
    [np.inf, -np.inf],
    np.nan,
)

X_extended = X_extended.fillna(
    X_extended.median(numeric_only=True)
)

In [44]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_los = extended_model_data["hospital_los_days"]

X_los_train, X_los_test, y_los_train, y_los_test = train_test_split(
    X_extended,
    y_los,
    test_size=0.2,
    random_state=42,
)

#LOS Model

In [50]:
y_los_log = np.log1p(
    extended_model_data["hospital_los_days"]
)

X_los_train, X_los_test, y_los_train, y_los_test = train_test_split(
    X_extended,
    y_los_log,
    test_size=0.2,
    random_state=42,
)

In [51]:
los_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42,
)

los_model.fit(X_los_train, y_los_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [52]:
los_pred_log = los_model.predict(X_los_test)

los_pred = np.expm1(los_pred_log)
y_los_test_days = np.expm1(y_los_test)

print(
    f"MAE: "
    f"{mean_absolute_error(y_los_test_days, los_pred):.2f} days"
)

print(
    f"RMSE: "
    f"{mean_squared_error(y_los_test_days, los_pred) ** 0.5:.2f} days"
)

MAE: 4.61 days
RMSE: 22.00 days


#Mortality Model

In [53]:
negative_count = (y_mort_train == 0).sum()
positive_count = (y_mort_train == 1).sum()

scale_pos_weight = negative_count / positive_count
scale_pos_weight

np.float64(83.21864951768488)

In [54]:
mortality_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
)

mortality_model.fit(
    X_mort_train,
    y_mort_train,
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [55]:
mortality_prob = mortality_model.predict_proba(
    X_mort_test
)[:, 1]

mortality_pred = (
    mortality_prob >= 0.20
).astype(int)

In [56]:
mortality_pred = mortality_model.predict(X_mort_test)

mortality_prob = mortality_model.predict_proba(
    X_mort_test
)[:, 1]

print(
    f"Accuracy : "
    f"{accuracy_score(y_mort_test, mortality_pred):.3f}"
)

print(
    f"Precision: "
    f"{precision_score(y_mort_test, mortality_pred):.3f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_mort_test, mortality_pred):.3f}"
)

print(
    f"F1 Score : "
    f"{f1_score(y_mort_test, mortality_pred):.3f}"
)

print(
    f"ROC AUC  : "
    f"{roc_auc_score(y_mort_test, mortality_prob):.3f}"
)

Accuracy : 0.852
Precision: 0.061
Recall   : 0.797
F1 Score : 0.114
ROC AUC  : 0.901


In [57]:
import joblib

In [58]:
joblib.dump(
    xgb_model,
    "../models/icu_admission_model.joblib"
)

joblib.dump(
    los_model,
    "../models/length_of_stay_model.joblib"
)

joblib.dump(
    mortality_model,
    "../models/mortality_model.joblib"
)

joblib.dump(
    scaler,
    "../models/scaler.joblib"
)

['../models/scaler.joblib']

In [59]:
from pathlib import Path

for file in Path("../models").glob("*.joblib"):
    print(file.name)

icu_admission_model.joblib
length_of_stay_model.joblib
mortality_model.joblib
scaler.joblib


In [60]:
joblib.dump(
    X.columns.tolist(),
    "../models/feature_columns.joblib"
)

['../models/feature_columns.joblib']

In [61]:
import joblib

joblib.dump(
    X.columns.tolist(),
    "../models/icu_feature_columns.joblib",
)

joblib.dump(
    X_extended.columns.tolist(),
    "../models/extended_feature_columns.joblib",
)

['../models/extended_feature_columns.joblib']

In [62]:
reference_values = {
    "sex": sorted(
        operations["sex"].dropna().astype(str).unique().tolist()
    ),
    "race": sorted(
        operations["race"].dropna().astype(str).unique().tolist()
    ),
    "department": sorted(
        operations["department"].dropna().astype(str).unique().tolist()
    ),
    "anesthesia_type": sorted(
        operations["antype"].dropna().astype(str).unique().tolist()
    ),
    "procedure_codes": sorted(
        operations["icd10_pcs"].dropna().astype(str).unique().tolist()
    ),
}

In [63]:
joblib.dump(
    reference_values,
    "../models/reference_values.joblib",
)

['../models/reference_values.joblib']